# Scoring a realistic clinical-interview transcript

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kikolo3000/clinical-skill-architecture/blob/main/notebooks/03_real_transcript.ipynb)

The first two notebooks used short single-utterance examples. This one runs the agent on a **15-turn synthetic interview** designed to look and feel like a real clinical conversation — multiple subscales, distractors, follow-ups. The transcript below is *not* a real patient; it is a hand-crafted illustration.

Total runtime ~6 minutes, total cost ~$0.10.

In [ ]:
# %pip install "clinical-skill-architecture[viz]"
import getpass, os
if not os.environ.get('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Anthropic API key: ')

TRANSCRIPT = '''\
I: thanks for coming in today. how have things been since we last spoke
S: honestly i don't really know where to start. some days are okay i guess but most of the time i just feel like nothing is moving. i can't seem to get out of bed before noon
I: that sounds really hard. can you tell me more about the mornings
S: it's like my body just won't cooperate. everything feels heavy. and even when i do get up i can't focus on anything. work has been impossible
I: have you been able to talk to anyone about this
S: not really. my partner tries but i feel like nobody actually understands. i don't even bother explaining anymore. they have their own stuff going on
I: are you sleeping at night
S: i fall asleep okay but i wake up at three or four in the morning and just lie there. my chest feels tight a lot of the time too. i went to my gp last month and they said it was probably stress
I: how are you feeling about the future right now
S: i don't see things getting better honestly. i used to be excited about applying to grad school but i can't even imagine doing that anymore. it just feels pointless
I: do you ever have thoughts of harming yourself
S: no nothing like that. i'm not going to do anything. it's more that i wish i could just stop thinking for a while
I: that distinction matters. i hear you. is there anything that has felt like a small bright spot lately
S: my dog i guess. she still gets me to go outside. and last weekend i made it to my sister's birthday which was nice even though i didn't say much
'''
print(TRANSCRIPT)

In [ ]:
from csa import score
result = score(TRANSCRIPT, model='claude-sonnet-4-6')

print(f'Total depression score:  {result.total_depression:.2f}')
print(f'Total words:             {result.word_count}')
print(f'Fragments scored:        {len(result.fragments)}')
print(f'Codings produced:        {len(result.codings)}')
print(f'Estimated cost:          ${result.estimated_cost_usd:.4f}')
print()
print('Per-subscale breakdown:')
for sub, s in result.subscales.items():
    flag = '●' if s.count > 0 else '·'
    print(f'  {flag} {sub} {s.name:24s}  score={s.score:5.2f}  count={s.count}  weighted_sum={s.weighted_sum}')

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

# Per-fragment heatmap of subscale weights
subscales = list(result.subscales.keys())
matrix = []
frag_labels = []
for fr in result.fragments:
    row = [fr.weighted_sum(s) for s in subscales]
    matrix.append(row)
    frag_labels.append(f'{fr.fragment.id}\n({fr.word_count}w)')

fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(matrix))))
im = ax.imshow(matrix, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(subscales))); ax.set_xticklabels(subscales)
ax.set_yticks(range(len(frag_labels))); ax.set_yticklabels(frag_labels, fontsize=8)
ax.set_title('Per-fragment weighted sum by subscale')
for i, row in enumerate(matrix):
    for j, v in enumerate(row):
        if v > 0:
            ax.text(j, i, str(v), ha='center', va='center', fontsize=9, color='black' if v < 4 else 'white')
fig.colorbar(im, ax=ax, label='weighted sum')
plt.tight_layout(); plt.show()

In [ ]:
# Print every coding with its rationale
for c in result.codings:
    print(f'[{c.subscale}/{c.sub_item}] persp={c.perspective} weight={c.weight}')
    print(f'  "{c.clause}"')
    print(f'  → {c.rationale}\n')

## Interpretation

Read the bar chart, the heatmap, and the per-clause rationales together. The agent should have flagged:
- **HOP** — `nothing is moving`, `i don't see things getting better`, `it just feels pointless`
- **PMR** — `everything feels heavy`, `can't focus on anything`, the lie-in-bed pattern
- **SOM** — `chest feels tight`, the early-morning waking
- **SAC** — the explicit denial of self-harm should be coded `denial` perspective (weight 1)
- **SEP** — `nobody actually understands`, `they have their own stuff going on`

The bright spots in the last turn (dog, sister's birthday) act as distractors and should *not* trigger codings.

## Use it on your own data

```python
from csa import score
result = score(open('my_transcript.md').read())
result.to_dict()    # dump to JSON for downstream analysis
```

Or batch:

```bash
for f in transcripts/*.md; do
    csa-score "$f" --json "runs/$(basename $f .md).json" --csv "runs/$(basename $f .md).csv"
done
```

Always read `DISCLAIMER.md` before scoring real patient data.